## Uses dataframe and verb/word locations to add word spans for label studio

Uses koondkorpus_examples.csv data file.

In [1]:
import pandas as pd
from estnltk import Text
from estnltk.taggers.system.rule_taggers.extraction_rules.ruleset import Ruleset
from estnltk.taggers.system.rule_taggers.extraction_rules.static_extraction_rule import StaticExtractionRule
from estnltk.taggers.system.rule_taggers.taggers.substring_tagger import SubstringTagger
#from estnltk.converters.label_studio.labelling_configurations import PhraseTaggingConfiguration
#from estnltk.converters.label_studio.labelling_tasks import PhraseTaggingTask
from estnltk.converters.label_studio.labelling_configurations import PhraseClassificationConfiguration
from estnltk.converters.label_studio.labelling_tasks import PhraseClassificationTask

## Configuration


In [2]:
SOURCE_DIR = "../source_data"
DATA_FILE = f"{SOURCE_DIR}/koondkorpus_examples.csv"
DATA_FILE2 = f"{SOURCE_DIR}/koondkorpus_examples2.csv"
DATA_FILE3 = f"{SOURCE_DIR}/koondkorpus_examples3.csv"

## Read in data

In [3]:
data1 = pd.read_csv(DATA_FILE, sep=";", encoding="utf-8")
data1 = data1.fillna("")
data2 = pd.read_csv(DATA_FILE2, sep=";", encoding="utf-8")
data2 = data2.fillna("")
data3 = pd.read_csv(DATA_FILE3, sep=";", encoding="utf-8")
data3 = data3.fillna("")

df = pd.concat([data1,data2, data3], axis=0).reset_index()

## Workflow

### EstNLTK task

Praeguses lahenduses on verb, comp ja käände põhjal tehtud groupby.

Laused on liidetud reavahetusega üheks tekstiks ning mitu lauset on ühena märgendamiseks ette antud.

See versioon panustab sellele, et ükski lause ei kordu ja sama käändega sõna ei esine mitmes lauses.

Esimese verbi näide demonstreerib, mis juhtub kui samas lauses on mitu verb+nimisõna esinemist.

Teine variant on toodud allpool.

In [4]:
data = df.groupby(['verb', 'verb_compound', 'kaane']).agg({
'verb_form': list,
'root_form': list,
'verb_loc': list,
'root_loc': list,
'sentence': list
}).reset_index()

In [5]:
data

,verb,verb_compound,kaane,verb_form,root_form,verb_loc,root_loc,sentence
0,kukkuma,alla,el,"[kukkus, kukkus]","[peast, laest]","[6, 6]","[8, 15]","[Mäletan , et aasta tagasi kukkus heast peast suures toas minu silme ees plafoon laest alla , tuhandeks killuks ., Mäletan , et aasta tagasi kukkus heast peast suures toas minu silme ees plafoon laest alla , tuhandeks killuks .]"
1,saama,,el,"[saab, saa, saa, saa, saab, saanud, sai]","[Minust, sõnadest, inimestest, toidupoest, tüdrukust, alkoholismist, sõprusest]","[3, 7, 3, 4, 32, 23, 3]","[2, 4, 5, 5, 31, 25, 5]","[“ Minust saab teenija ., Kui inimene ikka sõnadest aru ei saa , siis pole ka rusikatega midagi teha ., Ma ei saa aru inimestest , kes üldse napsu ei võta , nagu ka neist , kes hommikust õhtuni joovad ., Kui sa ei saa toidupoest soovitut , on asi päris hull ., Mulle millegipärast tundub , et mehed murravad truudust külmema südamega , sest nende jaoks on armukese juures tähtis see , et jõuaks käbe voodisse ja pärast on suva , mis tüdrukust saab ., “ Mu suhted on seni kõik läbi kukkunud , ” pihib BEATRICE , kes on uue armastuse , DJ PRIIT KUUSIKu kõrval saanud üle alkoholismist ., See suhe sai alguse sõprusest .]"
2,tulema,,adit,"[tulen, tuleb, tuled, tulnud]","[koju, koju, toime, kontserdimajja]","[6, 12, 5, 12]","[5, 13, 4, 14]","[Niipea , kui ma koju tulen , muutub ta tujukaks ja võimukaks : “ Tee seda !, Milline seksuaalfantaasia tundub kõige apetiitsem : seks tundmatuga liftis ; torulukksepp tuleb koju ; jõuluvana ja snegurotška või doktor ja patsient ?, Kuidas sa rahaliselt toime tuled ?, Mäe on kohtunud piletikassa juures pärnakatega , kes väidavad end olevat tulnud uude kontserdimajja juba üheksandat-kümnendat korda .]"


In [6]:
sentences = []

for i in range(len(data)):
    sentence = Text("\n".join(data.iloc[i]["sentence"]))
    #print(sentence)
    #break
    
    verbs = list(set(data.iloc[i]["verb_form"]))
    comps = [data.iloc[i]["verb_compound"]]
    roots = list(set(data.iloc[i]["root_form"]))

    rules1 = []
    for v in verbs:
        rules1.append(StaticExtractionRule(v, {'label': 'verb'}))
    for c in comps:
        rules1.append(StaticExtractionRule(c, {'label': 'verb_comp'}))
    for r in roots:
        rules1.append(StaticExtractionRule(r, {'label': 'nimisõna'}))
    
    rules = Ruleset(rules1)
    tagger = SubstringTagger(rules, output_attributes=['label'], ignore_case=True)
    tagger.tag(sentence)
    sentences.append(sentence)
    sentence.terms.display()


Mäletan , et aasta tagasi kukkus heast peast suures toas minu silme ees plafoon laest alla , tuhandeks killuks . Mäletan , et aasta tagasi kukkus heast peast suures toas minu silme ees plafoon laest alla , tuhandeks killuks .

“ Minust saab teenija . Kui inimene ikka sõnadest aru ei saa , siis pole ka rusikatega midagi teha . Ma ei saa aru inimestest , kes üldse napsu ei võta , nagu ka neist , kes hommikust õhtuni joovad . Kui sa ei saa toidupoest soovitut , on asi päris hull . Mulle millegipärast tundub , et mehed murravad truudust külmema südamega , sest nende jaoks on armukese juures tähtis see , et jõuaks käbe voodisse ja pärast on suva , mis tüdrukust saab . “ Mu suhted on seni kõik läbi kukkunud , ” pihib BEATRICE , kes on uue armastuse , DJ PRIIT KUUSIKu kõrval saanud üle alkoholismist . See suhe sai alguse sõprusest .

Niipea , kui ma koju tulen , muutub ta tujukaks ja võimukaks : “ Tee seda ! Milline seksuaalfantaasia tundub kõige apetiitsem : seks tundmatuga liftis ; torulukksepp tuleb koju ; jõuluvana ja snegurotška või doktor ja patsient ? Kuidas sa rahaliselt toime tuled ? Mäe on kohtunud piletikassa juures pärnakatega , kes väidavad end olevat tulnud uude kontserdimajja juba üheksandat-kümnendat korda .

In [10]:
#conf = PhraseTaggingConfiguration(['verb', 'verb_comp' 'nimisõna'], header="Verbobl ja nimisõnafraas")
#task = PhraseTaggingTask(conf, input_layer=tagger.output_layer, output_layer='annotated_partofspeech', label_attribute='label')

In [7]:
conf = PhraseClassificationConfiguration(phrase_labels=['verb', 'verb_comp', 'nimisõna'], 
                                         class_labels={'verb': 'verb', 'verb_comp':'verb_comp', 'nimisõna': 'nimisõna'},
                                         header="Vali analüüsitava sõna sõnaliik", 
                                         header_placement='middle')


In [8]:
task = PhraseClassificationTask(conf, input_layer=tagger.output_layer, output_layer=tagger.output_layer, label_attribute='label')

In [9]:
print(task.interface_file)

<View>
  <Labels name="phrase" toName="text" >
    <Label value="verb" background="#1b9e77" />
    <Label value="verb_comp" background="#d95f02" />
    <Label value="nimisõna" background="#7570b3" />
  </Labels>
  <Text name="text" value="$text" />
  <Header value="Vali analüüsitava sõna sõnaliik" />
  <Choices name="phrase_class" toName="text" choice="single-radio" >
    <Choice value="verb" alias="verb" />
    <Choice value="verb_comp" alias="verb_comp" />
    <Choice value="nimisõna" alias="nimisõna" />
  </Choices>
</View>


In [10]:
print(task.export_data(sentences, indent=2))

[
  {
    "data": {
      "text": "M\u00e4letan , et aasta tagasi kukkus heast peast suures toas minu silme ees plafoon laest alla , tuhandeks killuks .\nM\u00e4letan , et aasta tagasi kukkus heast peast suures toas minu silme ees plafoon laest alla , tuhandeks killuks ."
    },
    "annotations": [
      {
        "result": [
          {
            "value": {
              "start": 26,
              "end": 32,
              "labels": [
                "verb"
              ]
            },
            "from_name": "phrase",
            "to_name": "text",
            "type": "labels"
          },
          {
            "value": {
              "start": 39,
              "end": 44,
              "labels": [
                "nimis\u00f5na"
              ]
            },
            "from_name": "phrase",
            "to_name": "text",
            "type": "labels"
          },
          {
            "value": {
              "start": 80,
              "end": 85,
              "labels": [

Save label studio data file

In [11]:
with open('labelstudio/koondkorpus_examples_v1.json', 'w') as f:
    f.write( task.export_data(sentences, indent=2) )

## Teine meetod: leida iga sõna esinemise algus ja lõpp char ning selle põhjal märkida sõnu

### TODO: verb compound peale märkimine

Igale verbile ja sõnale on span, millele liidetakse eelneva lause pikkus, et lausete liitmisel spanid oleks õiged.

Label Studiosse importimiseks kasutatakse custom meetodit.

In [4]:
df["sent_lenght"] = df["sentence"].str.len()

In [5]:
data = df.groupby(['verb', 'verb_compound', 'kaane']).agg({
'verb_form': list,
'root_form': list,
'verb_loc': list,
'root_loc': list,
'sentence': list,
'verb_span': list,
'root_span': list,
'sent_lenght': list,
}).reset_index()

Lisa igale spanile eelnevate lauste pikkused, et spanid jääks õigeks.

In [8]:
data["ilus_osa_enne"] = data["verb"] + " " + data["verb_compound"] + " ("+ data["kaane"] + ")\n\n" 

In [17]:
new_verb_spans = []
new_root_spans = []

for i in range(len(data)):
    prev_sent_lens = data.iloc[i]["sent_lenght"]
    verb_spans_to_change = data.iloc[i]["verb_span"]
    root_spans_to_change = data.iloc[i]["root_span"]
    
    block_verb_spans = []
    block_root_spans = []
    prev_sent_sum = len(data.iloc[i]["ilus_osa_enne"])
    for j in range(len(verb_spans_to_change)):
        parts = verb_spans_to_change[j].replace("(", "").replace(")", "").split(",")
        new_start = int(parts[0])+prev_sent_sum
        new_end =  int(parts[1])+prev_sent_sum
        block_verb_spans.append((new_start, new_end))
        
        parts = root_spans_to_change[j].replace("(", "").replace(")", "").split(",")
        new_start = int(parts[0])+prev_sent_sum
        new_end =  int(parts[1])+prev_sent_sum
        block_root_spans.append((new_start, new_end))
        
        prev_sent_sum+=prev_sent_lens[j]+1 # +1 sest hiljem liidetakse laused reavahetusega kokku
            
    new_verb_spans.append(block_verb_spans)
    new_root_spans.append(block_root_spans)

In [18]:
data["new_verb_span"] = new_verb_spans
data["new_root_span"] = new_root_spans

In [19]:
data

,verb,verb_compound,kaane,verb_form,root_form,verb_loc,root_loc,sentence,verb_span,root_span,sent_lenght,new_verb_span,new_root_span,ilus_osa_enne
0,kukkuma,alla,el,"[kukkus, kukkus]","[peast, laest]","[6, 6]","[8, 15]","[Mäletan , et aasta tagasi kukkus heast peast suures toas minu silme ees plafoon laest alla , tuhandeks killuks ., Mäletan , et aasta tagasi kukkus heast peast suures toas minu silme ees plafoon laest alla , tuhandeks killuks .]","[(26, 32), (26, 32)]","[(39, 44), (80, 85)]","[112, 112]","[(45, 51), (158, 164)]","[(58, 63), (212, 217)]",kukkuma alla (el)\n\n
1,saama,,el,"[saab, saa, saa, saa, saab, saanud, sai]","[Minust, sõnadest, inimestest, toidupoest, tüdrukust, alkoholismist, sõprusest]","[3, 7, 3, 4, 32, 23, 3]","[2, 4, 5, 5, 31, 25, 5]","[“ Minust saab teenija ., Kui inimene ikka sõnadest aru ei saa , siis pole ka rusikatega midagi teha ., Ma ei saa aru inimestest , kes üldse napsu ei võta , nagu ka neist , kes hommikust õhtuni joovad ., Kui sa ei saa toidupoest soovitut , on asi päris hull ., Mulle millegipärast tundub , et mehed murravad truudust külmema südamega , sest nende jaoks on armukese juures tähtis see , et jõuaks käbe voodisse ja pärast on suva , mis tüdrukust saab ., “ Mu suhted on seni kõik läbi kukkunud , ” pihib BEATRICE , kes on uue armastuse , DJ PRIIT KUUSIKu kõrval saanud üle alkoholismist ., See suhe sai alguse sõprusest .]","[(9, 13), (33, 36), (6, 9), (10, 13), (182, 186), (107, 113), (9, 12)]","[(2, 8), (17, 25), (14, 24), (14, 24), (172, 181), (118, 131), (20, 29)]","[23, 76, 98, 55, 188, 133, 31]","[(22, 26), (70, 73), (120, 123), (223, 226), (451, 455), (565, 571), (601, 604)]","[(15, 21), (54, 62), (128, 138), (227, 237), (441, 450), (576, 589), (612, 621)]",saama (el)\n\n
2,tulema,,adit,"[tulen, tuleb, tuled, tulnud]","[koju, koju, toime, kontserdimajja]","[6, 12, 5, 12]","[5, 13, 4, 14]","[Niipea , kui ma koju tulen , muutub ta tujukaks ja võimukaks : “ Tee seda !, Milline seksuaalfantaasia tundub kõige apetiitsem : seks tundmatuga liftis ; torulukksepp tuleb koju ; jõuluvana ja snegurotška või doktor ja patsient ?, Kuidas sa rahaliselt toime tuled ?, Mäe on kohtunud piletikassa juures pärnakatega , kes väidavad end olevat tulnud uude kontserdimajja juba üheksandat-kümnendat korda .]","[(21, 26), (90, 95), (27, 32), (73, 79)]","[(16, 20), (96, 100), (21, 26), (85, 99)]","[75, 152, 34, 133]","[(37, 42), (182, 187), (272, 277), (353, 359)]","[(32, 36), (188, 192), (266, 271), (365, 379)]",tulema (adit)\n\n


In [10]:
from pd_collection_to_ls import collection_to_labelstudio, conf_gen

In [11]:
with open(f"labelstudio/koondkorpus_examples.txt", "w", encoding="utf-8") as f:
    f.write(conf_gen(classes=["V","OBL"]))

Custom label studio import kood. "multi" viitab sellele, et ühe märgendatava tekstina on mitu lauset.

In [20]:
collection_to_labelstudio(data, state="multi",filename="labelstudio/koondkorpus_examples_v2.json")